<a href="https://colab.research.google.com/github/sofia-henat/Proyecto2/blob/main/Proyecto2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto 2**

Integrantes:

* Hernandez Nativitas Sofia Alejandra
* Ortiz Briseño Emiliano
* Ruiz Ramírez Santiago

# 1. Preparación y lectura de datos

Aquí lo que hacemos es importar las herramientas que usaremos (NumPy para matemáticas y Matplotlib para dibujar por ejemplo). También incluimos una función que lea los números que se escriban y automáticamente añade el conjugado si es que se ingresa un número complejo (para mantener los coeficientes reales).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, clear_output

def parse_roots(roots_str):
    """Convierte texto en números complejos y añade conjugadas automáticamente."""
    if not roots_str.strip():
        return []
    roots = []
    try:
        raw_roots = [complex(r.strip()) for r in roots_str.split(',')]
        for r in raw_roots:
            roots.append(r)
            if r.imag != 0:
                roots.append(np.conj(r))
    except ValueError:
        pass
    return roots

# 2. Cálculos y generación de gráficas
Aquí es donde tomamos los ceros, polos y ganancia para calcular la ecuación del sistema. Tras esto verificamos si el sistema es estable y generar las cuatro gráficas solicitadas, la del plano Z, la magnitud, la fase y la respuesta al impulso.

In [ ]:
def plot_system(zeros_str, poles_str, K, w_val):
    zeros = parse_roots(zeros_str)
    poles = parse_roots(poles_str)

    # Construir coeficientes
    # Aquí es prácticamente donde hacemos la función de transferencia
    b = K * np.poly(zeros) if len(zeros) > 0 else [K]
    a = np.poly(poles) if len(poles) > 0 else [1.0]

    def get_factor_string(roots):
        if not roots:
            return "1"
        # Redondeamos a 3 decimales para que no se vea amontonado
        return " * ".join([f"(z - {np.round(r, 3)})" for r in roots])

    num_str = get_factor_string(zeros)
    den_str = get_factor_string(poles)

    print("-" * 50)
    print(f"FORMA FACTORIZADA H(z):")
    print(f"H(z) = {K} * [ {num_str} ] / [ {den_str} ]")
    print("-" * 50)

    print("Numerador (b):", np.round(b, 4))
    print("Denominador (a):", np.round(a, 4))

    is_stable = all(abs(p) < 1.0 for p in poles)
    print(f"Estabilidad: {'ESTABLE' if is_stable else 'INESTABLE'} (Polos < 1)")

    fig = plt.figure(figsize=(15, 10))

    # 1. Plano Z
    ax1 = plt.subplot(2, 2, 1)
    ax1.add_patch(plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--'))
    ax1.axhline(0, color='black', lw=1)
    ax1.axvline(0, color='black', lw=1)

    if poles:
        ax1.scatter(np.real(poles), np.imag(poles), s=100, marker='x', color='red', label='Polos')
    if zeros:
        ax1.scatter(np.real(zeros), np.imag(zeros), s=100, marker='o', facecolors='none', edgecolors='blue', label='Ceros')

    point_w = np.exp(1j * w_val)
    ax1.scatter(np.real(point_w), np.imag(point_w), color='green', s=100, label=f'e^(j{w_val:.2f})')

    dist_text = []
    for i, z in enumerate(zeros):
        ax1.plot([np.real(point_w), np.real(z)], [np.imag(point_w), np.imag(z)], 'b--', alpha=0.5)
        dist_text.append(f"D_z{i}: {abs(point_w - z):.2f}")
    for i, p in enumerate(poles):
        ax1.plot([np.real(point_w), np.real(p)], [np.imag(point_w), np.imag(p)], 'r--', alpha=0.5)
        dist_text.append(f"D_p{i}: {abs(point_w - p):.2f}")

    ax1.set_xlim([-2, 2])
    ax1.set_ylim([-2, 2])
    ax1.set_aspect('equal')
    ax1.set_title("Plano Z y Exploración Geométrica")
    ax1.legend(loc='upper right')
    if dist_text:
        ax1.text(-1.9, 1.9, "\n".join(dist_text), verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))

    # Aquí es donde evaluamos H(z) --> z = e^jomega
    w, h = signal.freqz(b, a, worN=512, whole=False)

    # 2. Magnitud
    ax2 = plt.subplot(2, 2, 2)
    ax2.plot(w, np.abs(h), 'b')
    ax2.axvline(w_val, color='green', linestyle='--', label=f'w = {w_val:.2f}')
    ax2.set_title("Respuesta en Frecuencia (Magnitud)")
    ax2.grid(True)

    # 3. Fase
    ax3 = plt.subplot(2, 2, 3)
    ax3.plot(w, np.angle(h), 'b')
    ax3.axvline(w_val, color='green', linestyle='--', label=f'w = {w_val:.2f}')
    ax3.set_title("Respuesta en Frecuencia (Fase)")
    ax3.grid(True)

    # 4. Respuesta al impulso
    ax4 = plt.subplot(2, 2, 4)
    n = np.arange(50)
    impulse = np.zeros(50)
    impulse[0] = 1.0
    h_n = signal.lfilter(b, a, impulse)
    ax4.stem(n, h_n)
    ax4.set_title("Respuesta al Impulso h[n]")
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

# 3. Controles de la interfaz interactiva
En esta parte es dónde dibujamos las cajas de texto y el deslizador en la pantalla de Colab. Luego, conectamos estos controles con la función principal para que cada vez que modifiquemos un número, las gráficas se actualicen por sí mismas.

In [ ]:
# Configuración de botones y controles
zeros_input = widgets.Text(value='', description='Ceros:', style={'description_width': 'initial'})
poles_input = widgets.Text(value='', description='Polos:', style={'description_width': 'initial'})
gain_input = widgets.FloatText(value=1.0, description='Ganancia K:')
w_slider = widgets.FloatSlider(value=0, min=0, max=np.pi, step=0.05, description='w (rad):')

# Vincular controles con la gráfica
widgets.interact(plot_system, zeros_str=zeros_input, poles_str=poles_input, K=gain_input, w_val=w_slider);

"""
Paso bajas:
* Polos = 0.8, Ceros = -1
Paso altas:
* Polos = 0.8, Ceros = 1
Notch:
* Polos = 0+1j, Ceros = 0+0.9j
Resonancia y estabilidad
* Polos = 0.5+0.5j, Polos = 0.9+0.9j
"""

interactive(children=(Text(value='', description='Ceros:', style=DescriptionStyle(description_width='initial')…

'\nPaso bajas:\n* Polos = 0.8, Ceros = -1\nPaso altas:\n* Polos = 0.8, Ceros = 1\nNotch:\n* Polos = 0+1j, Ceros = 0+0.9j\nResonancia y estabilidad\n* Polos = 0.5+0.5j, Polos = 0.9+0.9j\n'